# CRISPRon: reconstruct LuoSpCas92020_min200 inputs and create a fixed split

This notebook starts directly from the official `Luo2020_Kim2019.xlsx`
workbook and uses **only** rows where:

```text
Dataset == "LuoSpCas92020_min200"
```

The expected subset contains **10,543 samples**.

For this Luo-only subset:

- `30mer_gRNA` is used as the sequence input;
- `Indel_freq_HEK293T` is used as the final rescaled activity label and copied to
  `Quant_norm_efficiency`;
- the missing scalar `CRISPRoff` feature is generated from the 23-mer
  protospacer+PAM using the official CRISPRoff pipeline.

The standardized benchmark split is then created:

- 64% training
- 16% validation
- 20% unseen
- seed = 42


In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path(".").resolve()

WORKBOOK = Path("Luo2020_Kim2019.xlsx")
DATA_SHEET = "Luo2020_Kim2019"

# These paths follow the CRISPRon README.
CRISPROFF_SCRIPT = Path("bin/CRISPRspec_CRISPRoff_pipeline.py")
ENERGY_DICS = Path("data/model/energy_dics.pkl")

OUTPUT_DIR = Path("results/crispron_LuoSpCas92020_min200_fixed_split")
PREP_DIR = OUTPUT_DIR / "preprocessing"
SPLIT_DIR = OUTPUT_DIR / "saved_splits"
EXPORT_DIR = OUTPUT_DIR / "split_exports"

for directory in [OUTPUT_DIR, PREP_DIR, SPLIT_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42

if not WORKBOOK.exists():
    raise FileNotFoundError(
        f"Missing {WORKBOOK.resolve()}. Place Luo2020_Kim2019.xlsx "
        "in the CRISPRon repository root or update WORKBOOK."
    )

print("Workbook:", WORKBOOK.resolve())
print("CRISPRoff script:", CRISPROFF_SCRIPT.resolve())
print("Energy parameters:", ENERGY_DICS.resolve())


## Read and verify the official 23,902-guide workbook

In [ ]:
raw = pd.read_excel(
    WORKBOOK,
    sheet_name=DATA_SHEET,
)

expected_columns = {
    "30mer_gRNA",
    "Indel_freq_HEK293T",
    "Dataset",
}

missing = expected_columns.difference(raw.columns)

if missing:
    raise ValueError(
        f"Workbook is missing expected columns: {sorted(missing)}"
    )

# Use only the LuoSpCas92020_min200 subset.
data = raw.loc[
    raw["Dataset"].astype(str) == "LuoSpCas92020_min200"
].copy()

if len(data) == 0:
    raise ValueError(
        "No rows with Dataset == 'LuoSpCas92020_min200' were found."
    )

data["30mer_gRNA"] = (
    data["30mer_gRNA"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("U", "T", regex=False)
)

valid = (
    data["30mer_gRNA"]
    .str.fullmatch(r"[ACGT]{30}")
    .fillna(False)
    & pd.to_numeric(
        data["Indel_freq_HEK293T"],
        errors="coerce",
    ).notna()
)

if not valid.all():
    raise ValueError(
        f"{int((~valid).sum())} invalid Luo rows were found."
    )

if data["30mer_gRNA"].duplicated().any():
    raise ValueError(
        "The LuoSpCas92020_min200 subset unexpectedly contains duplicate 30-mers."
    )

data["Quant_norm_efficiency"] = pd.to_numeric(
    data["Indel_freq_HEK293T"],
    errors="raise",
)

data = data.reset_index(drop=True)

data["sample_id"] = np.arange(
    len(data),
    dtype=int,
)

print("Filtered dataset:", "LuoSpCas92020_min200")
print("Rows:", len(data))
print("Unique 30-mers:", data["30mer_gRNA"].nunique())
print(
    "Quant_norm_efficiency range:",
    float(data["Quant_norm_efficiency"].min()),
    "to",
    float(data["Quant_norm_efficiency"].max()),
)

EXPECTED_LUO_ROWS = 10543

if len(data) != EXPECTED_LUO_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_LUO_ROWS} LuoSpCas92020_min200 samples, "
        f"but found {len(data)}."
    )


## Generate the exact 23-mer input required by CRISPRoff

CRISPRon defines the 30-mer as:

```text
4-nt upstream + 20-nt protospacer + 3-nt PAM + 3-nt downstream
```

Therefore the CRISPRoff guide sequence is:

```python
30mer[4:27]
```

which is the 20-nt protospacer plus 3-nt PAM.


In [ ]:
guides_fasta = PREP_DIR / "LuoSpCas92020_min200_10543_23mers.fa"

with guides_fasta.open(
    "w",
    encoding="utf-8",
) as handle:
    for sample_id, sequence_30mer in zip(
        data["sample_id"].to_numpy(dtype=int),
        data["30mer_gRNA"].astype(str).to_numpy(),
    ):
        sequence_23mer = sequence_30mer[4:27]

        if len(sequence_23mer) != 23:
            raise RuntimeError(
                "Generated 23-mer has unexpected length."
            )

        handle.write(
            f">sample_{sample_id}\n"
            f"{sequence_23mer}\n"
        )

print(
    "Generated:",
    guides_fasta.resolve(),
)
print(
    "Number of guides:",
    len(data),
)


## Calculate CRISPRoff scores

If `CRISPRoff_results.tsv` has already been created, this cell reuses it.

Otherwise it calls the same CRISPRoff program used by the CRISPRon pipeline
with `--no_azimuth` and `--guide_params_out`.

Before running this cell, the CRISPRon README requires:

```text
bin/CRISPRspec_CRISPRoff_pipeline.py
data/model/energy_dics.pkl
```

to be present.


In [ ]:
from pathlib import Path
import sys

RNAFOLD = Path(
    "/opt/anaconda3/envs/crispron_venv/bin/RNAfold"
)

print("Python:", sys.executable)
print("RNAfold exists:", RNAFOLD.exists())
print("RNAfold:", RNAFOLD)

In [ ]:
crisproff_results = (
    PREP_DIR
    / "CRISPRoff_results.tsv"
)

specificity_report = (
    PREP_DIR
    / "CRISPRspec.tsv"
)

needs_crisproff_run = (
    not crisproff_results.exists()
    or crisproff_results.stat().st_size == 0
)

if needs_crisproff_run:
    # Remove incomplete outputs from a previous failed run.
    if crisproff_results.exists():
        crisproff_results.unlink()

    if specificity_report.exists():
        specificity_report.unlink()

    if not CRISPROFF_SCRIPT.exists():
        raise FileNotFoundError(
            "CRISPRspec_CRISPRoff_pipeline.py is missing:\n"
            f"{CRISPROFF_SCRIPT.resolve()}\n\n"
            "Download CRISPRoff and copy "
            "CRISPRspec_CRISPRoff_pipeline.py into the CRISPRon bin/ folder."
        )

    if not ENERGY_DICS.exists():
        raise FileNotFoundError(
            "energy_dics.pkl is missing:\n"
            f"{ENERGY_DICS.resolve()}\n\n"
            "Place energy_dics.pkl at the path configured by ENERGY_DICS."
        )

    RNAFOLD = Path(
        "/opt/anaconda3/envs/crispron_venv/bin/RNAfold"
    )
    
    if not RNAFOLD.exists():
        raise FileNotFoundError(
            f"RNAfold was not found at: {RNAFOLD}"
        )

    command = [
        sys.executable,
        str(CRISPROFF_SCRIPT),
        "--guides",
        str(guides_fasta),
        "--specificity_report",
        str(specificity_report),
        "--guide_params_out",
        str(crisproff_results),
        "--duplex_energy_params",
        str(ENERGY_DICS),
        "--rnafold_x",
        str(RNAFOLD),
        "--no_azimuth",
    ]

    print("Running CRISPRoff command:\n")
    print(" ".join(command))

    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
    )

    print("\n===== STDOUT =====")
    print(completed.stdout)

    print("\n===== STDERR =====")
    print(completed.stderr)

    print("\nReturn code:")
    print(completed.returncode)

    if completed.returncode != 0:
        raise RuntimeError(
            "CRISPRoff failed. Inspect STDERR above for the underlying error."
        )

else:
    print(
        "Reusing existing non-empty CRISPRoff file:",
        crisproff_results.resolve(),
    )

# Strict output checks before parsing.
if not crisproff_results.exists():
    raise RuntimeError(
        "CRISPRoff did not create CRISPRoff_results.tsv."
    )

file_size = crisproff_results.stat().st_size

print(
    "\nCRISPRoff result file size:",
    file_size,
    "bytes",
)

if file_size == 0:
    raise RuntimeError(
        "CRISPRoff_results.tsv was created but is empty. "
        "The CRISPRoff command did not complete correctly."
    )

# Make sure the file contains at least one non-comment, non-empty line.
with crisproff_results.open(
    "r",
    encoding="utf-8",
    errors="replace",
) as handle:
    informative_lines = [
        line
        for line in handle
        if line.strip()
        and not line.lstrip().startswith("#")
    ]

if len(informative_lines) == 0:
    raise RuntimeError(
        "CRISPRoff_results.tsv contains no parseable non-comment lines."
    )

print(
    "CRISPRoff output passed the non-empty-content check."
)


## Parse and merge `CRISPRoff_score`

In [ ]:
from pandas.errors import EmptyDataError

try:
    crisproff = pd.read_csv(
        crisproff_results,
        sep="\t",
        comment="#",
    )
except EmptyDataError as exc:
    raise RuntimeError(
        "CRISPRoff_results.tsv exists but contains no parseable table. "
        "Rerun the previous CRISPRoff cell and inspect its STDERR output."
    ) from exc

if crisproff.empty:
    raise RuntimeError(
        "CRISPRoff_results.tsv was parsed successfully but contains zero data rows."
    )

print("CRISPRoff columns:")
print(crisproff.columns.tolist())
print("CRISPRoff rows:", len(crisproff))

column_lookup = {
    str(column).strip().lower(): column
    for column in crisproff.columns
}

guide_id_column = None
score_column = None

for candidate in [
    "guideid",
    "guide_id",
    "id",
]:
    if candidate in column_lookup:
        guide_id_column = column_lookup[
            candidate
        ]
        break

for candidate in [
    "crisproff_score",
    "crisproff",
    "crisproffscore",
]:
    if candidate in column_lookup:
        score_column = column_lookup[
            candidate
        ]
        break

if guide_id_column is None:
    raise ValueError(
        "Could not identify the guide-ID column in CRISPRoff output. "
        f"Columns were: {crisproff.columns.tolist()}"
    )

if score_column is None:
    raise ValueError(
        "Could not identify CRISPRoff_score in output. "
        f"Columns were: {crisproff.columns.tolist()}"
    )

crisproff_small = crisproff[
    [
        guide_id_column,
        score_column,
    ]
].copy()

crisproff_small.columns = [
    "guideID",
    "CRISPRoff",
]

crisproff_small["sample_id"] = (
    crisproff_small["guideID"]
    .astype(str)
    .str.extract(
        r"sample_(\d+)",
        expand=False,
    )
)

if crisproff_small[
    "sample_id"
].isna().any():
    raise ValueError(
        "At least one CRISPRoff guide ID did not match sample_<number>."
    )

crisproff_small["sample_id"] = (
    crisproff_small[
        "sample_id"
    ].astype(int)
)

crisproff_small["CRISPRoff"] = (
    pd.to_numeric(
        crisproff_small[
            "CRISPRoff"
        ],
        errors="raise",
    )
)

if crisproff_small[
    "sample_id"
].duplicated().any():
    raise ValueError(
        "Duplicate sample IDs were found in CRISPRoff output."
    )

model_data = data.merge(
    crisproff_small[
        [
            "sample_id",
            "CRISPRoff",
        ]
    ],
    on="sample_id",
    how="left",
    validate="one_to_one",
)

if model_data[
    "CRISPRoff"
].isna().any():
    raise ValueError(
        f"{int(model_data['CRISPRoff'].isna().sum())} samples "
        "are missing CRISPRoff values after merging."
    )

print("Final modeling rows:", len(model_data))
print(
    "CRISPRoff range:",
    float(model_data["CRISPRoff"].min()),
    "to",
    float(model_data["CRISPRoff"].max()),
)

if len(model_data) != 10543:
    raise ValueError(
        f"Expected 10,543 fully reconstructed Luo samples, but found {len(model_data)}."
    )

model_data.to_csv(
    PREP_DIR
    / "CRISPRon_LuoSpCas92020_min200_modeling_table.csv",
    index=False,
)


## Create the fixed 64/16/20 benchmark split

In [ ]:
N = len(model_data)

np.random.seed(SEED)

seen_indices = np.random.choice(
    N,
    size=int(0.80 * N),
    replace=False,
)

unseen_indices = np.setdiff1d(
    np.arange(N),
    seen_indices,
)

train_indices, validation_indices = (
    train_test_split(
        seen_indices,
        test_size=0.20,
        random_state=SEED,
    )
)

train_data = model_data.iloc[
    train_indices
].reset_index(drop=True)

validation_data = model_data.iloc[
    validation_indices
].reset_index(drop=True)

unseen_data = model_data.iloc[
    unseen_indices
].reset_index(drop=True)

assert set(train_indices).isdisjoint(
    validation_indices
)
assert set(train_indices).isdisjoint(
    unseen_indices
)
assert set(validation_indices).isdisjoint(
    unseen_indices
)

display(
    pd.DataFrame(
        {
            "subset": [
                "train",
                "validation",
                "unseen",
            ],
            "n_samples": [
                len(train_data),
                len(validation_data),
                len(unseen_data),
            ],
            "fraction": [
                len(train_data) / N,
                len(validation_data) / N,
                len(unseen_data) / N,
            ],
        }
    )
)


## Save splits and manifest

In [ ]:
train_file = (
    SPLIT_DIR
    / "train_split.csv"
)
validation_file = (
    SPLIT_DIR
    / "validation_split.csv"
)
unseen_file = (
    SPLIT_DIR
    / "unseen_split.csv"
)

train_data.to_csv(
    train_file,
    index=False,
)

validation_data.to_csv(
    validation_file,
    index=False,
)

unseen_data.to_csv(
    unseen_file,
    index=False,
)

np.savetxt(
    OUTPUT_DIR
    / "train_indices.txt",
    train_indices,
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR
    / "validation_indices.txt",
    validation_indices,
    fmt="%d",
)

np.savetxt(
    OUTPUT_DIR
    / "unseen_indices.txt",
    unseen_indices,
    fmt="%d",
)

for name, frame in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    folder = EXPORT_DIR / name
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    frame["30mer_gRNA"].to_csv(
        folder
        / f"{name}_30mer_gRNA.txt",
        index=False,
        header=False,
    )

    frame[
        "Quant_norm_efficiency"
    ].to_csv(
        folder
        / f"{name}_Quant_norm_efficiency.txt",
        index=False,
        header=False,
    )

    frame["CRISPRoff"].to_csv(
        folder
        / f"{name}_CRISPRoff.txt",
        index=False,
        header=False,
    )


def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


manifest = {
    "selected_dataset": "LuoSpCas92020_min200",
    "source_workbook": str(
        WORKBOOK.resolve()
    ),
    "source_sheet": DATA_SHEET,
    "sequence_column": "30mer_gRNA",
    "activity_source_column": (
        "Indel_freq_HEK293T"
    ),
    "training_activity_column": (
        "Quant_norm_efficiency"
    ),
    "activity_reconstruction": (
        "Quant_norm_efficiency is copied from the workbook's "
        "final merged/rescaled Indel_freq_HEK293T column."
    ),
    "crisproff_source": str(
        crisproff_results.resolve()
    ),
    "crisproff_column": (
        "CRISPRoff_score"
    ),
    "preprocessing_modified": False,
    "dataset_filter": "LuoSpCas92020_min200",
    "split_seed": SEED,
    "total_samples": int(N),
    "expected_total_samples": 10543,
    "n_train": int(
        len(train_data)
    ),
    "n_validation": int(
        len(validation_data)
    ),
    "n_unseen": int(
        len(unseen_data)
    ),
    "train_sha256": sha256(
        train_file
    ),
    "validation_sha256": sha256(
        validation_file
    ),
    "unseen_sha256": sha256(
        unseen_file
    ),
}

with open(
    OUTPUT_DIR
    / "split_manifest.json",
    "w",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

print(
    "Saved to:",
    OUTPUT_DIR.resolve(),
)
